In [1]:
import numpy as np
import pandas as pd

from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.metrics import rand_score, mutual_info_score, fowlkes_mallows_score
from sklearn.metrics import adjusted_rand_score, adjusted_mutual_info_score
from sklearn.metrics import normalized_mutual_info_score
from sklearn.metrics import homogeneity_score, completeness_score, v_measure_score

import matplotlib.pyplot as plt

import warnings

warnings.filterwarnings('ignore')

np.random.seed(42)

In [2]:
df = pd.read_csv("delivery_locations.csv")
df.head()

,delivery_location_latitude,delivery_location_longitude,delivery_zone
0,40.7247,-74.0081,Tribeca
1,40.7556,-74.0018,Hell's Kitchen
2,40.7695,-73.9882,Hell's Kitchen
3,40.7476,-73.9954,Chelsea
4,40.7101,-74.0061,Financial District


In [3]:
zones_map = {
    "Financial District": 0,
    "SoHo": 1,
    "Chelsea": 2, 
    "Midtown": 3,
    "Upper East Side": 4,
    "Greenwich Village": 5,
    "Gramercy": 6,
    "Tribeca": 7,
    "Hell's Kitchen": 8,
    "Upper West Side": 9
}

df["zones_label"] = df["delivery_zone"].map(zones_map)

In [4]:
scaler = StandardScaler()
df[["scaled_latitude", "scaled_longitude"]] = scaler.fit_transform(df[["delivery_location_latitude", "delivery_location_longitude"]])
df.head()

,delivery_location_latitude,delivery_location_longitude,delivery_zone,zones_label,scaled_latitude,scaled_longitude
0,40.7247,-74.0081,Tribeca,7,-0.829529,-1.089614
1,40.7556,-74.0018,Hell's Kitchen,8,0.471412,-0.693579
2,40.7695,-73.9882,Hell's Kitchen,8,1.056625,0.161353
3,40.7476,-73.9954,Chelsea,2,0.134599,-0.291258
4,40.7101,-74.0061,Financial District,0,-1.444213,-0.963889


In [ ]:
model = GaussianMixture(n_components=10)
df["clusters"] = model.fit_predict(df[["scaled_latitude", "scaled_longitude"]])

  File "c:\Users\upeks\AppData\Local\Programs\Python\Python312\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "c:\Users\upeks\AppData\Local\Programs\Python\Python312\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\upeks\AppData\Local\Programs\Python\Python312\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "c:\Users\upeks\AppData\Local\Programs\Python\Python312\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


In [6]:
latitudes = df["delivery_location_latitude"].to_list()
longitudes = df["delivery_location_longitude"].to_list()

In [7]:
fig, ax = plt.subplots()

scatter = ax.scatter(latitudes, longitudes, c=df["clusters"].to_list())

# produce a legend with the unique colors from the scatter
legend1 = ax.legend(*scatter.legend_elements(),
                    loc="best", title="Clusters")
ax.add_artist(legend1)
ax.set_ylabel("Longitudes")
ax.set_xlabel("Latitudes")

plt.savefig("plot.png", dpi=300, bbox_inches="tight")
plt.close()

In [8]:
sl = silhouette_score(df[["scaled_latitude", "scaled_longitude"]], df["clusters"])
db = davies_bouldin_score(df[["scaled_latitude", "scaled_longitude"]], df["clusters"])
ch = calinski_harabasz_score(df[["scaled_latitude", "scaled_longitude"]], df["clusters"])

r = rand_score(df["zones_label"].to_list(), df["clusters"].to_list())
mi = mutual_info_score(df["zones_label"].to_list(), df["clusters"].to_list())
fm = fowlkes_mallows_score(df["zones_label"].to_list(), df["clusters"].to_list())

ar = adjusted_rand_score(df["zones_label"].to_list(), df["clusters"].to_list())
ami = adjusted_mutual_info_score(df["zones_label"].to_list(), df["clusters"].to_list())

nmi = normalized_mutual_info_score(df["zones_label"].to_list(), df["clusters"].to_list())

h = homogeneity_score(df["zones_label"].to_list(), df["clusters"].to_list())
c = completeness_score(df["zones_label"].to_list(), df["clusters"].to_list())
vm = v_measure_score(df["zones_label"].to_list(), df["clusters"].to_list())

In [9]:
pd.DataFrame(
    {
        "Silhouette Score": [sl],
        "Davies-Bouldin Index": [db],
        "Calinski-Harabasz Index": [ch],
        "Rand index": [r],
        "Mutual Information": [mi],
        "Fowlkes-Mallows Index": [fm],
        "Adjusted Rand Index": [ar],
        "Adjusted Mutual Information": [ami],
        "Normalized Mutual Information": [nmi],
        "Homogeneity": [h],
        "Completeness ": [c],
        "V-measure": [vm],
        
    }
).to_csv("metrics.csv", index=False)